In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from fowt_ml import Config
from fowt_ml.datasets import get_data_mfiles
from fowt_ml.pipeline import Pipeline
from IPython.display import display
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

sys.path.append("../")
from utils import DATA_DECAY_CL, RECORDINGS_CSV, correct_force

# --- Split strategy -------------------------------------------------------
# Group = condition (dof + ic). All repeats of a condition stay together, so the
# model never sees a near-identical twin of a test recording during training.
# Do *not* shuffle rows, and do *not* group by experiment_id (repeats would leak; high frequency for the former, repetition for the latter).
#
# Note: ic1 is roughly the tail of ic2 within a DOF, so this split is lenient
# (tests "same DOF, new amplitude"). Harder checks to add afterwards:
#   1. leave-one-DOF-out (groups = dof); if skillful then implies has learnt general physics
#   2. train on ic1, test on ic2 (real amplitude extrapolation)
#
# Restrict to the 16xx experiment batch (mooring == "lin") for now, as in the paper; exclude exp1649
# until confirmed (is it erroneous? it doesn't look like a normal decay oscillation).
# Never combine decay_cl with decay_cl_test (identical copies).
#
# Inputs: ACCt, ACCb, POS. Target: Ft (6 outputs).
# Metrics: RMSE and R2 per output, per fold. Tune inside training folds only.
#
# There remains an open question of how much of the notebook content can be moved into FOWT-ML/Pipeline.
# --------------------------------------------------------------------------

In [ ]:

# constants

# `decay_cl` holds the full 16xx batch; `decay_cl_test` holds identical copies of some of its files, so never load both
DATA_FOLDER = DATA_DECAY_CL
KHZ_SCALE = 1e3  # scale factor; kHz to seconds

# per-recording dof, ic, mooring and repeat from the test diary
# N.B. `inventory/` is a local working area (excluded from git)
MANUAL_NOTES_CSV = Path("../inventory/manual_notes.csv")

# only the linear mooring (16xx batch) for now, as this is what the paper (Taruffi & Viré, 2026) uses
# the 12xx batch (`qs` mooring, 3 repeats per condition) is left out until we decide whether the batches can be combined
MOORING = "lin"

# exp1649 (heave ic2) is suspect until confirmed, so heave only has ic1 in this batch
EXCLUDE_RECORDINGS = [1649]

# 12 conditions in "lin" setup, one recording each (6 dof x 2 ic, minus heave ic2, plus pitch ic2* from exp1659 at ic=10deg)
# 2 are held out as the final test set, the other 10 go into CV with ~2 per fold
# N.B. `GroupKFold` keeps conditions whole but balances folds by row count, not by number of conditions,
# so a long recording can fill a fold alone whilst short ones get packed together (1-3 conditions per fold);
# per-fold scores then mix arbitrary motions and conditions aren't weighted equally in the mean
# `LeaveOneGroupOut` (one condition per fold, ~2x the fits) gives one clean score per condition,
# but pitch_ic2/ic2* would need a shared group first, otherwise the near-twin always sits in training
# and would need to consider whether want to be testing on the question of: 'how well predict a new condition having never seen it' or more generic than that?
N_TEST_CONDITIONS = 2
N_FOLDS = 5

# accelerometer columns used by the physics baseline (for `correct_force` function)
ACC_TT_COLS = [f"acc_tt_meas3[{i}]" for i in range(3)]
ACC_TB_COLS = [f"acc_tb_meas3[{i}]" for i in range(3)]
PHYSICS_BASELINE = "PhysicsBaseline"

## 1 Model selection

### Create a config

In [ ]:
# load the config
config_file = "./experiment_force_correction.yml"
config = Config.from_yaml(config_file)

ml_setup = config["ml_setup"]

In [ ]:
# set the model names; we selected three models
# TODO: hyperparameters are as of yet untouched
# TODO: can do some tuning within the pipeline/on the validation set?
! TODO: use something like Optuna/Ray Tune for hyperparameter tuning?
ml_setup["model_names"] = {
    "LinearRegression": {},
    "MultilayerPerceptron": {
        "hidden_layer_sizes": (8, 8, 8, 8, 8, 8),
        "activation": "relu",
        "max_iter": 500,
        "alpha": 0.00001,
        "learning_rate": "adaptive",
        "validation_fraction": 0.15,
        "early_stopping": True,
        "n_iter_no_change": 15,
        "random_state": 42,
    },
    "XGBoostRegression": {
        "n_estimators": 500,
        "max_depth": 10,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "colsample_bylevel": 0.7,
        "min_child_weight": 5,
        "gamma": 0.0,
        "tree_method": "hist",
        "random_state": 42,
    },
}

In [ ]:
! this relates to the inner split that `Pipeline.setup()` does, which is not used for scoring here
# `Pipeline.setup()` always runs its own `train_test_split` and fits only on the "train" part, so 20% of whatever we pass in is discarded
# all splitting that matters is done here instead (hold-out + grouped CV), and the scores `compare_models` returns from this inner split are ignored
# shuffle so the discarded 20% is spread over all conditions; without it, the tail of the data (i.e. whole conditions) would never be trained on
# N.B. the usual shuffle leak doesn't matter here, as nothing is scored on the inner split
# TODO: drop this inner split once FOWT-ML supports grouped/pre-defined splits (https://github.com/hybridlabs-nl/FOWT-ML/issues/50)
ml_setup["train_test_split_kwargs"] = {
    "train_size": 0.80,
    "shuffle": True,
    "random_state": 42,
}

In [ ]:

# closed-lopp, surge, sway, heave, roll, pitch, yaw, no wind, no-rotation
# data_ids = ["exp1644", "exp1646", "exp1648", "exp1651", "exp1653", "exp1656"]

### Compare models with different data

In [ ]:
# get all data
! this opens all exp in the folder and brings them into one dataframe
! added the aux recording data, so each row knows its experiment
df = get_data_mfiles(
    DATA_FOLDER, aux_csv_file=RECORDINGS_CSV, aux_df_column_name="recording_number"
)

# TODO: is this useful?! Last data gets to e.g. 2052 days, which misrepresents the time scales of the concat'd data (not one timeseries)
# TODO: needs to be time per experiment, not across experiments
df.index = pd.to_timedelta(df.index * KHZ_SCALE, unit="s")  # set time index

# attach dof, ic, mooring and repeat from the test diary, so each row knows its condition
notes = pd.read_csv(MANUAL_NOTES_CSV)
notes.index = notes["experiment_id"].str.removeprefix("exp").astype(int)
for col in ["diary_dof", "diary_ic", "mooring", "rep"]:
    df[col] = df["recording_number"].map(notes[col])

# restrict to the lin batch, drop suspect recordings, and keep only the columns used below
keep_rows = (df["mooring"] == MOORING) & ~df["recording_number"].isin(
    EXCLUDE_RECORDINGS
)
keep_cols = ["recording_number", "time", "diary_dof", "diary_ic", "rep"]
df = df.loc[keep_rows, keep_cols + ml_setup["predictors"] + ml_setup["targets"]].copy()

# group = condition (dof + ic), so all repeats of a condition land in the same fold
df["condition"] = df["diary_dof"] + "_" + df["diary_ic"]
groups = df["condition"].to_numpy()

# sanity check: expect 12 conditions with one recording each
display(
    df.groupby("condition")["recording_number"]
    .unique()
    .apply(sorted)
    .to_frame("recordings")
)

! data passed to the pipeline is only predictors and targets, no aux data
! this is because setup procedure currently donesn't handle non-numeric data in aux; see vault notes
# TODO: this subsetting shouldn't be needed when/if addressed upstream
ML_COLS = ml_setup["predictors"] + ml_setup["targets"]  # all numeric, no NaNs
df_ml = df[ML_COLS]

### Hold out test conditions

In [ ]:
# hold out whole conditions (all repeats), untouched until the final evaluation
# N.B. random pick, so the test set may hold both ics of one dof (that dof would then be entirely unseen)
holdout = GroupShuffleSplit(n_splits=1, test_size=N_TEST_CONDITIONS, random_state=42)
trainval_idx, test_idx = next(holdout.split(df_ml, groups=groups))
df_trainval, df_test = df.iloc[trainval_idx], df.iloc[test_idx]
groups_trainval = groups[trainval_idx]
test_conditions = sorted(set(groups[test_idx]))
print(f"hold-out (test) conditions: {test_conditions}")

In [ ]:
def per_output_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Return RMSE and R2 for each output (3 forces and 3 torques)."""
    return pd.DataFrame(
        {
            "output": ml_setup["targets"],
            "rmse": root_mean_squared_error(y_true, y_pred, multioutput="raw_values"),
            "r2": r2_score(y_true, y_pred, multioutput="raw_values"),
        }
    )


! N.B. this may need closer inspection/checking at some point.. I haven't really looked at it much
def physics_baseline_ft(data: pd.DataFrame) -> np.ndarray:
    """Return the physics-based estimate of Ft, i.e. the non-aerodynamic load from `correct_force`.

    With no wind and no rotor rotation the aerodynamic load is zero, so measured Ft minus the
    estimated aerodynamic load is the physics model's prediction of Ft.

    N.B. the `correct_force` coefficients come from a regression in the original work, which may
    have used some of these recordings, so the baseline isn't necessarily out-of-sample.
    """
    ft = data[ml_setup["targets"]].to_numpy()
    faero_est, _ = correct_force(
        data[ACC_TT_COLS].to_numpy(), data[ACC_TB_COLS].to_numpy(), ft
    )
    return ft - faero_est

### Grouped cross-validation on the train/val conditions

In [ ]:
# no shuffling, so fold membership is deterministic
splitter = GroupKFold(n_splits=N_FOLDS)

fold_metrics = []  # one row per fold x model x output
oof_predictions = []  # out-of-fold predictions, one frame per validation fold

! N.B. a lot of the (grouped) cross validation procedure is done here outside FOWT-ML/Pipeline
# TODO: this is something that could be moved upstream, so the pipeline can handle grouped CV and return the scores and predictions for each fold?
for fold, (train_idx, val_idx) in enumerate(
    splitter.split(df_trainval, groups=groups_trainval)
):
    df_train, df_val = df_trainval.iloc[train_idx], df_trainval.iloc[val_idx]
    val_conditions = sorted(set(groups_trainval[val_idx]))
    print(f"fold {fold}: validating on {val_conditions}")

    # fit every model on the training conditions only
    # N.B. any hyperparameter tuning has to happen in here too (e.g. a grouped split of `df_train`), never on `df_val`
    # TODO: there's a TODO in the FOWT-ML.compare_models() to parallise the comparison, which would be useful here...
    fold_pipeline = Pipeline(config)
    fold_pipeline.setup(
        data=df_train[ML_COLS]
    )  #! FYI, at this point, the pipeline will do its own train/test split on df_train, meaning we are not using the full df_train
    models, _ = fold_pipeline.compare_models(sort="r2")

    # predict the validation conditions, plus the physics baseline on the same rows
    X_val = df_val[ml_setup["predictors"]].to_numpy()
    Y_val = df_val[ml_setup["targets"]].to_numpy()
    predictions = {name: model.predict(X_val) for name, model in models.items()}
    predictions[PHYSICS_BASELINE] = physics_baseline_ft(df_val)

    # score each model on this fold
    for name, Y_pred in predictions.items():
        fold_metrics.append(
            per_output_metrics(Y_val, Y_pred).assign(
                model=name, fold=fold, val_conditions=", ".join(val_conditions)
            )
        )

    # keep the predictions, labelled with their recording, condition and fold
    oof = df_val[["recording_number", "time", "condition", "rep"]].assign(fold=fold)
    oof[ml_setup["targets"]] = Y_val
    for name, Y_pred in predictions.items():
        oof[[f"{name}:{t}" for t in ml_setup["targets"]]] = Y_pred
    oof_predictions.append(oof)

fold_metrics = pd.concat(fold_metrics, ignore_index=True)
oof_predictions = pd.concat(oof_predictions)

In [ ]:
# per output, mean and spread across folds
summary = fold_metrics.groupby(["output", "model"])[["rmse", "r2"]].agg(["mean", "std"])
display(summary)

In [ ]:
# R2 per fold and output, models side by side
fold_metrics.pivot_table(
    index=["fold", "val_conditions", "output"], columns="model", values="r2"
)

In [ ]:
# CV scores per model, averaged over folds and outputs, used to pick the best model
cv_scores = (
    fold_metrics.groupby("model")[["rmse", "r2"]]
    .mean()
    .sort_values("r2", ascending=False)
)
best_model = cv_scores.drop(index=PHYSICS_BASELINE).index[0]
display(cv_scores)
print(f"best model (mean CV R2): {best_model}")

### Final evaluation on the hold-out conditions

In [ ]:
# refit every model on all train/val conditions, then score once on the untouched test conditions
final_pipeline = Pipeline(config)
final_pipeline.setup(data=df_trainval[ML_COLS])
models, _ = final_pipeline.compare_models(sort="r2")

# predict the test conditions, plus the physics baseline on the same rows
X_test = df_test[ml_setup["predictors"]].to_numpy()
Y_test = df_test[ml_setup["targets"]].to_numpy()
test_predictions = {name: model.predict(X_test) for name, model in models.items()}
test_predictions[PHYSICS_BASELINE] = physics_baseline_ft(df_test)

test_metrics = pd.concat(
    [
        per_output_metrics(Y_test, Y_pred).assign(model=name)
        for name, Y_pred in test_predictions.items()
    ],
    ignore_index=True,
)
display(test_metrics.pivot(index="output", columns="model", values=["rmse", "r2"]))

### Save results for the dashboard

In [ ]:
# save what `ml_dashboard_cv.ipynb` needs, so the results can be explored there without re-training
# N.B. here, `inventory/` is a local working area (excluded from git) and will be user specific
! this is a tmp solution, workflows upstream in fowt-ml could support this (see https://github.com/hybridlabs-nl/FOWT-ML/issues/50)
import json
from datetime import datetime

import joblib

RESULTS_DIR = Path("../inventory/ml_results_cv")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# hold-out rows, labelled with their experiment and per-experiment time
test_data = df_test[["recording_number", "time", "condition", "rep"] + ML_COLS]
test_data.reset_index(drop=True).to_parquet(RESULTS_DIR / "test_data.parquet")

# final models (refit on all train/val conditions) and their CV scores
joblib.dump(models, RESULTS_DIR / "models.joblib")
cv_scores.to_csv(RESULTS_DIR / "scores.csv", index_label="model")

# CV detail: per fold x model x output metrics, and out-of-fold predictions
fold_metrics.to_csv(RESULTS_DIR / "fold_metrics.csv", index=False)
oof_predictions.reset_index(drop=True).to_parquet(
    RESULTS_DIR / "oof_predictions.parquet"
)

# run settings, so the dashboard can say what it's showing
run_info = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "data_folder": str(DATA_FOLDER),
    "mooring": MOORING,
    "excluded_recordings": EXCLUDE_RECORDINGS,
    "recordings": sorted(int(r) for r in df["recording_number"].unique()),
    "n_rows": len(df_ml),
    "n_trainval": len(df_trainval),
    "n_test": len(df_test),
    "holdout_split": f"GroupShuffleSplit(test_size={N_TEST_CONDITIONS}, random_state=42) by condition: {test_conditions}",
    "cv_split": f"GroupKFold(n_splits={N_FOLDS}), groups = condition (dof + ic)",
    "folds": fold_metrics.groupby("fold")["val_conditions"].first().to_dict(),
    "pipeline_split": ml_setup["train_test_split_kwargs"],
    "predictors": ml_setup["predictors"],
    "targets": ml_setup["targets"],
    "metric_names": ["rmse", "r2"],
    "model_names": ml_setup["model_names"],
    "best_model": best_model,
}
with open(RESULTS_DIR / "run_info.json", "w") as f:
    json.dump(run_info, f, indent=2, default=str)
print(f"Saved results to {RESULTS_DIR.resolve()}")